In [106]:
"""
Constructs the DFT of the dihedral group in SageMath using knowledge of the representation theory.

TO-DO: work over finite field.

notes: # Gershgorin's theorem did not provide good bounds for the eigenvalues.

""";

In [119]:
# ── Configuration ──────────────────────────────────────────────────────────────
n = 5; print("n =", n)
USE_FINITE_FIELD = False
q = 11  # prime power; ignored when USE_FINITE_FIELD = False

n = 5


In [120]:
# ── Build the coefficient ring and choose omega ─────────────────────────────────
if USE_FINITE_FIELD:
    # Find the smallest k such that n | (q^k - 1),
    # i.e. the multiplicative order of q mod n.
    k = Zmod(n)(q).multiplicative_order()
    F = GF(q**k, 'a')
    print(f"Working in GF({q}^{k}) = GF({q**k})")

    # Find a primitive n-th root of unity in F.
    # The multiplicative group of GF(q^k) is cyclic of order q^k - 1,
    # so a generator g satisfies g^((q^k-1)/n) has order n.
    g = F.multiplicative_generator()
    omega = g**((q**k - 1) // n)
    assert omega**n == F.one(), "omega is not an n-th root of unity"
    assert omega**(n-1) != F.one() or n == 1, "omega is not primitive"
    print(f"omega = {omega}  (order {omega.multiplicative_order()})")
else:
    # Symbolic / complex mode (original behaviour)
    var('z')
    forget()
    assume(z**n == 1)
    omega = z
    F = None
    print(f"omega = {omega}  (symbolic)")

omega = z  (symbolic)


In [121]:
G = DihedralGroup(n); print("G =", G) #D_n, dihedral group of order 2n
r = [g for g in G if g.order() == n][0] #rotation of order n, generator
s = [g for g in G if g.order() == 2 and g != r**(n//2)][0] #flip of order 2, generator
print("r =", r)
print("s =", s)

G = Dihedral group of order 10 as a permutation group
r = (1,5,4,3,2)
s = (2,5)(3,4)


In [122]:
# returns (0, k) if g = r^k and (1, k) if g = s*r^k
def express_in_gens(g):
    for k in range(n):
        if g == r**k:
            return (0, k)
    for k in range(n):
        if g == s * r**k:
            return (1, k)

In [123]:
# n odd, we have two 1-dim'l irreps and (n-1)/2 2-dim'l irreps
# the 1-dim's irreps are trivial and sign
# the 2-dim'l irreps are given by rotation matrices and a flip matrix
def rho_odd(k, g, omega):
    (s_exp, r_exp) = express_in_gens(g)
    if k == 0:
        return matrix([1])
    if k == -1:
        if s_exp == 0:
            return matrix([1])
        if s_exp == 1:
            return matrix([-1])
    if k >= 1:
        if s_exp == 0:
            return matrix([[omega**(k*r_exp), 0], [0, omega**(-k*r_exp)]])
        if s_exp == 1:
            return matrix([[0, omega**(k*r_exp)], [omega**(-k*r_exp), 0]])

In [124]:
def dft_matrix_odd(omega):
    assert n % 2 == 1
    rows = []
    for g in G:
        row = [rho_odd(k, g, omega).list() for k in range(-1,(n-1)//2 + 1)]
        rows.append(sum(row, []))
    return matrix(rows)

In [125]:
# for n even case
def rho_even(k, g, omega):
    (s_exp, r_exp) = express_in_gens(g)
    if k == 0:   # trivial
        return matrix([1])
    if k == -1:  # sign of rotation
        return matrix([(-1)**r_exp])
    if k == -2:  # sign of reflection
        return matrix([(-1)**s_exp])
    if k == -3:  # total sign
        return matrix([(-1)**(r_exp + s_exp)])
    if k >= 1:
        if s_exp == 0:
            return matrix([[omega**(k*r_exp), 0], [0, omega**(-k*r_exp)]])
        if s_exp == 1:
            return matrix([[0, omega**(k*r_exp)], [omega**(-k*r_exp), 0]])

In [126]:
# form the DFT matrix for n even
def dft_matrix_even(omega):
    assert n % 2 == 0
    rows = []
    for g in G:
        row = [rho_even(k, g, omega).list() for k in range(-3, n//2)]
        rows.append(sum(row, []))
    return matrix(rows)

In [127]:
DFT_matrix = dft_matrix_odd(omega) if n % 2 == 1 else dft_matrix_even(omega); print(DFT_matrix)

[     1      1      1      0      0      1      1      0      0      1]
[     1      1      z      0      0    1/z    z^2      0      0 z^(-2)]
[     1      1    z^2      0      0 z^(-2)    z^4      0      0 z^(-4)]
[     1      1    z^3      0      0 z^(-3)    z^6      0      0 z^(-6)]
[     1      1    z^4      0      0 z^(-4)    z^8      0      0 z^(-8)]
[    -1      1      0      1      1      0      0      1      1      0]
[    -1      1      0      z    1/z      0      0    z^2 z^(-2)      0]
[    -1      1      0    z^2 z^(-2)      0      0    z^4 z^(-4)      0]
[    -1      1      0    z^3 z^(-3)      0      0    z^6 z^(-6)      0]
[    -1      1      0    z^4 z^(-4)      0      0    z^8 z^(-8)      0]


In [134]:
f = DFT_matrix.charpoly(); print(f)

x^10 + (-z^4 - z^2 - 1/z^6 - 2)*x^9 + (-2*z^7 + z^6 + 2*z^4 - z - 1/z - 1/z^2 - 1/z^3 + 2/z^6 - 1/z^12 + 2)*x^8 + (2*z^11 - z^10 + 2*z^9 - z^8 + 2*z^7 - 2*z^6 - 2*z^4 - 2*z^3 + 2*z^2 + z - 4/z + 4/z^2 + 1/z^3 + 1/z^5 - 4/z^6 + 4/z^8 + 1/z^9 + 1/z^10 - 1/z^11 + 2/z^12 + 1/z^18 - 1/z^19 - 6)*x^7 + (-2*z^13 + z^12 - 3*z^11 + z^10 - z^9 + z^8 + z^7 + 6*z^6 + 6*z^4 - 2*z^3 - 3*z^2 - 8*z + 4/z + 6/z^2 - 2/z^4 + 3/z^5 + 6/z^6 - 7/z^8 - 1/z^11 + 1/z^12 - 2/z^14 + 2/z^15 - 1/z^16 + 1/z^17 - 2/z^18 + 2/z^19 - 7)*x^6 + (z^16 - 2*z^15 + 3*z^13 - 2*z^12 + 3*z^11 - 2*z^10 - 8*z^9 + 5*z^8 + 6*z^7 - 11*z^6 + 6*z^5 + 2*z^4 + 2*z^3 + 10*z^2 - 11*z - 5/z - 1/z^2 + 13/z^3 - 3/z^4 + 1/z^5 + 4/z^6 - 8/z^8 - 1/z^10 - 4/z^11 + 2/z^12 + 8/z^13 + 2/z^14 - 5/z^15 + 2/z^16 - 3/z^17 + 2/z^18 - 2/z^19 + 2/z^20 - 2/z^21 + 1/z^22 - 5)*x^5 + (-z^18 + 4*z^17 - z^16 - 5*z^15 + 5*z^14 - 7*z^13 + 9*z^12 - 13*z^11 - z^10 + 17*z^9 - 15*z^8 + 26*z^7 + 4*z^6 - 6*z^5 - 28*z^4 - 7*z^3 + 9*z^2 - 4*z - 4/z + 3/z^2 + 3/z^3 + 16/z^

In [129]:
if USE_FINITE_FIELD:
    L.<a> = f.splitting_field(); L

In [138]:
if USE_FINITE_FIELD:
    eigenvalues = f.roots(L, multiplicities=False); eigenvalues

In [ ]:
def frobenius_orbit(alpha, p):
    orbit = []
    seen = set()
    
    x = alpha
    while x not in seen:
        seen.add(x)
        orbit.append(x)
        x = x^p
    
    return orbit

In [ ]:
def frobenius_orbits(eigenvalues, p):
    orbits = []
    seen = set()
    
    for lam in eigenvalues:
        if lam not in seen:
            orb = frobenius_orbit(lam, p)
            orbits.append(orb)
            seen.update(orb)
    
    return orbits

In [ ]:
def discrete_log_Fq(x, alpha=None):
    F = x.parent()
    if x == 0:
        raise ValueError("Log undefined for 0")
    if alpha is None:
        alpha = F.multiplicative_generator()
    return x.log(alpha)

In [ ]:
frobenius_orbits(eigenvalues, p)

[[8, 6, 7, 2],
 [3*a^7 + 4*a^6 + 6*a^4 + 5*a^3 + 6*a^2 + 4*a + 2,
  3*a^8 + 7*a^7 + 7*a^6 + 10*a^5 + 3*a^4 + 8*a^3 + 6*a^2 + 5*a + 2,
  7*a^8 + 5*a^7 + 7*a^6 + 8*a^4 + 5*a^3 + 4*a^2 + 3*a + 6,
  9*a^8 + 4*a^7 + 9*a^6 + a^5 + a^4 + 3*a^3 + 5*a^2 + 2*a + 6,
  9*a^8 + 10*a^6 + 2*a^4 + 8*a^3 + 4*a^2 + 3*a + 1,
  2*a^8 + 8*a^7 + 9*a^6 + 10*a^5 + 10*a^4 + 4*a^3 + 10*a^2 + 4*a + 6,
  7*a^8 + 10*a^7 + 4*a^6 + 10*a^5 + 4*a^4 + 10*a^3 + 9*a^2 + 3*a + 9,
  2*a^8 + 5*a^7 + 3*a^6 + 8*a^5 + 8*a^4 + 2*a^3 + 5*a^2 + 6*a + 7,
  4*a^8 + 5*a^7 + 5*a^6 + 6*a^5 + 8*a^4 + 6*a^3 + 9*a^2 + 4,
  2*a^8 + 8*a^7 + 7*a^6 + a^5 + a^4 + 3*a^2 + 3*a + 2,
  4*a^8 + a^7 + 7*a^6 + 5*a^5 + a^4 + 3*a^3 + 10*a^2 + 10*a + 10,
  2*a^8 + 5*a^7 + 6*a^6 + 5*a^5 + 3*a^4 + 6*a^3 + 5*a^2 + 7*a + 9,
  8*a^8 + 2*a^7 + 2*a^6 + 6*a^2 + 9*a + 1,
  4*a^8 + 10*a^6 + 5*a^4 + 5*a^3 + 3*a^2 + 4*a + 10,
  6*a^8 + 3*a^7 + 8*a^6 + 7*a^5 + a^3 + 3*a^2 + 7*a,
  a^8 + 3*a^7 + 2*a^6 + 3*a^4 + 9*a^3 + 7*a^2 + 5*a + 8,
  10*a^8 + 6*a^7 + 8*a^6 + 3*a

In [ ]:
dlogs = [discrete_log_Fq(x, alpha=None)/(L.order()-1) for x in eigenvalues]; dlogs

[3/10,
 87887147/168424835,
 58202932/168424835,
 134957747/168424835,
 90957167/168424835,
 137136537/168424835,
 124634442/168424835,
 23580182/168424835,
 161103227/168424835,
 158404662/168424835]

In [ ]:
DFT_matrix.eigenvalues()

[8,
 2*z9^8 + 6*z9^7 + 3*z9^6 + 5*z9^5 + 6*z9^4 + z9^3 + 7*z9^2 + z9 + 3,
 2*z9^8 + 6*z9^7 + 5*z9^6 + 7*z9^5 + 6*z9^4 + 2*z9^3 + 2*z9^2 + 2*z9 + 3,
 3*z9^8 + 9*z9^7 + 10*z9^6 + 4*z9^5 + 9*z9^4 + z9^3 + z9^2 + 6*z9 + 3,
 8*z9^8 + 7*z9^7 + 9*z9^6 + 6*z9^5 + 2*z9^3 + 8*z9^2 + z9 + 5,
 6*z9^8 + 9*z9^7 + z9^6 + 5*z9^5 + z9^4 + 4*z9^3 + 8*z9^2 + 8*z9 + 6,
 5*z9^8 + 6*z9^7 + 6*z9^6 + 9*z9^5 + 10*z9^4 + 9*z9^3 + 3*z9^2 + 6,
 6*z9^8 + 8*z9^7 + 6*z9^6 + z9^5 + 4*z9^4 + 9*z9^3 + z9^2 + 10,
 z9^8 + z9^7 + 7*z9^5 + 2*z9^4 + 8*z9^2,
 3*z9^7 + 4*z9^6 + 6*z9^4 + 5*z9^3 + 6*z9^2 + 4*z9 + 2]

In [ ]:
# can normalize by 1/sqrt(D) to get a unitary matrix
if not USE_FINITE_FIELD:
    print(DFT_matrix.conjugate_transpose() * DFT_matrix)

In [ ]:
# form norm polynomial by acting on coefficients of characteristic polynomial by the Galois group
def norm_poly(f, n):

    def sigma(i):
        phi = K.hom([z**i])
        return f.map_coefficients(phi)
    
    units = [i for i in range(1, n) if gcd(i, n) == 1]  # (Z/nZ)^×
    result = prod(sigma(i) for i in units)
    return result.change_ring(QQ)

In [ ]:
f = DFT_matrix.charpoly(); f

x^6 + (2*z - 1)*x^5 + (-6*z - 3)*x^4 + (5*z + 1)*x^3 + (-24*z - 9)*x^2 + (36*z + 36)*x - 54

In [ ]:
Nf = norm_poly(f, n); Nf

x^12 - 4*x^11 + 7*x^10 - 21*x^9 + 54*x^8 - 93*x^7 + 165*x^6 - 297*x^5 + 657*x^4 - 1026*x^3 + 972*x^2 - 1944*x + 2916

In [ ]:
Nf.is_irreducible()

True

In [ ]:
Nf.galois_group()

Transitive group number 299 of degree 12

In [ ]:
G = TransitiveGroup(12, 299)
print(G.order())
print(G.is_solvable())
print(G.is_primitive())
print(G.structure_description())

1036800
False
False
(A6 x A6) : D4
